This notebook performs webscraping to loop over the hrefs scraped previously and scrapes their product features.

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException
from webdriver_manager.chrome import ChromeDriverManager

import pandas as pd
import numpy as np
import re

In [2]:
options = Options()

#options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-infobars')
options.add_argument('--disable-extensions')
options.add_argument('--disable-notifications')
options.add_argument('--window-size=1724,968')
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/96.0.4664.110 Safari/537.36"
options.add_argument(f'user-agent={user_agent}')
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

In [3]:
# load product href csvs
red_df = pd.read_csv("../data/Red_wine_hrefs.csv")
white_df = pd.read_csv("../data/White_wine_hrefs.csv")
sparkling_df = pd.read_csv("../data/Champagne-Sparkling_wine_hrefs.csv")

# Combine the DataFrames into a single DataFrame
wine_df = pd.concat([red_df, white_df, sparkling_df], ignore_index=True)

In [4]:
wine_df

,href,type
0,https://www.danmurphys.com.au/product/DM_17275...,Red
1,https://www.danmurphys.com.au/product/DM_10535...,Red
2,https://www.danmurphys.com.au/product/DM_90074...,Red
3,https://www.danmurphys.com.au/product/DM_90104...,Red
4,https://www.danmurphys.com.au/product/DM_14446...,Red
...,...,...
6542,https://www.danmurphys.com.au/product/DM_13411...,Champagne-Sparkling
6543,https://www.danmurphys.com.au/product/DM_12199...,Champagne-Sparkling
6544,https://www.danmurphys.com.au/product/DM_10851...,Champagne-Sparkling
6545,https://www.danmurphys.com.au/product/DM_10750...,Champagne-Sparkling


## Webscraping
The following cell will perform scraping over the hrefs. The scraping takes a long time and if it fails before completing, the remaining cells of the notebook can be run to process and save the data that has been scraped. Set the starting_i variable in the following cell to the index of the href from which you would like to start scraping. This allows the process to be started where the previous run failed.
## The features to be scraped are:
### product description
### type (of grapes), separated by space
### vintage
### state
### region
### wine sweetness
### wine body
### winery name
### product brand
### product name
### non-member price
### member price

In [5]:
%%time

hrefs_subset= []
product_brands = []
product_names = []
colours = []
product_descriptions = []
types = []
vintages = []
states = []
regions = []
wineries = []
sizes = []
non_member_prices = []
member_prices = []
bodies = []
sweetnesses = []

total_wines = len(wine_df)
starting_i = 6544
try:
    for i in range(starting_i, len(wine_df), 1):
        
        # get href
        href = wine_df.iloc[i]["href"]
        #hrefs_subset.append(href)
        driver.get(href)
    
        #colours.append(wine_df.iloc[i]["type"])
        
        is_loaded = False
        try:
            # expand specifications list
            wait = WebDriverWait(driver, 10)
            expansion_panel = wait.until(EC.element_to_be_clickable((By.XPATH, "//mat-expansion-panel[contains(., 'Product specifications')]")))
            expansion_panel.click()
            is_loaded = True
        except TimeoutException:
            print("!!! no product page for i =", i, "--href", href)
    
        if is_loaded:    
            # Scrape product brand and name
            product_brands.append(driver.find_element(By.CLASS_NAME, "product-title__brand").get_attribute("innerText"))
            product_names.append(driver.find_element(By.CLASS_NAME, "product-title__name").get_attribute("innerText"))
        
        
            # scrape product description
            product_descriptions.append(driver.find_element(By.CLASS_NAME, "product-skeleton__details-description-reg").get_attribute("innerText"))
            
            
            # scrape specifications list
            specification_ul = driver.find_element(By.CLASS_NAME, "product-attribute__items")
            specification_li_elements = specification_ul.find_elements(By.TAG_NAME, "li")
            specifications = {}
            for li in specification_li_elements:
                key = li.find_element(By.CLASS_NAME, "product-attribute__item-key").get_attribute("innerText")
                value = li.find_element(By.CLASS_NAME, "product-attribute__item-value").get_attribute("innerText")
                specifications[key] = value
            
            if "Type" in specifications:
                types.append(specifications["Type"])
            else:
                types.append(np.nan)
            
            if "Current Vintage" in specifications:
                vintages.append(specifications["Current Vintage"])
            else:
                vintages.append(np.nan)
            
            if "State" in specifications:
                states.append(specifications["State"])
            else:
                states.append(np.nan)
            
            if "Region" in specifications:
                regions.append(specifications["Region"])
            else:
                regions.append(np.nan)
            
            if "Wine Maker" in specifications:
                wineries.append(specifications["Wine Maker"])
            else:
                wineries.append(np.nan)
            
            if "Size" in specifications:
                sizes.append(specifications["Size"])
            else:
                sizes.append(np.nan)
        
            if "Wine Sweetness" in specifications:
                sweetnesses.append(specifications["Wine Sweetness"])
            else:
                sweetnesses.append(np.nan)
        
            if "Wine Body" in specifications:
                bodies.append(specifications["Wine Body"])
            else:
                bodies.append(np.nan)
        
        
            # Scrape prices
            pattern = r'\d+\.*\d*' # regex pattern
            price_contect = driver.find_element(By.ID, "mat-tab-content-1-0")
            try:
                non_member_price = price_contect.find_element(By.CLASS_NAME, "pack__price-offer").get_attribute("innerText")
                non_member_price = re.search(pattern, non_member_price).group()
                member_price = price_contect.find_element(By.CLASS_NAME, "pack__price").get_attribute("innerText")
                member_price = re.search(pattern, member_price).group()
            except NoSuchElementException:
                try:
                    non_member_price = price_contect.find_element(By.CLASS_NAME, "pack__price").get_attribute("innerText")
                    non_member_price = re.search(pattern, non_member_price).group()
                    member_price = np.nan
                    print("no member price")
                except NoSuchElementException:
                    non_member_price = np.nan
                    member_price = np.nan
                    print("no price found for: i =", i, href)
            
            non_member_prices.append(non_member_price)
            member_prices.append(member_price)
        
        else:
            product_brands.append(np.nan)
            product_names.append(np.nan)
            product_descriptions.append(np.nan)
            types.append(np.nan)
            vintages.append(np.nan)
            states.append(np.nan)
            regions.append(np.nan)
            wineries.append(np.nan)
            sizes.append(np.nan)
            non_member_prices.append(np.nan)
            member_prices.append(np.nan)
            bodies.append(np.nan)
            sweetnesses.append(np.nan)  
            
        hrefs_subset.append(href)
        colours.append(wine_df.iloc[i]["type"])
        print("scraped i =", i, "of", total_wines-1)

    last_i_scraped = i

except Exception as e:
    print("Stopped scraping at i =", i)
    print(e)
    

no member price
scraped i = 6544 of 6546
!!! no product page for i = 6545 --href https://www.danmurphys.com.au/product/DM_107506/perrier-jouet-blanc-de-blanc-champagne?isFromSearch=false&isPersonalised=false&pageName=category_page
scraped i = 6545 of 6546
!!! no product page for i = 6546 --href https://www.danmurphys.com.au/product/DM_106404/cannibal-creek-blanc-de-blanc?isFromSearch=false&isPersonalised=false&pageName=category_page
scraped i = 6546 of 6546
CPU times: user 152 ms, sys: 2.94 ms, total: 155 ms
Wall time: 27.5 s


In [6]:
cut_off = i - starting_i # make sure all lists are the same length

populated_wine_df = {
    "href" : hrefs_subset[:cut_off],
    "product_brand" : product_brands[:cut_off],
    "product_name" : product_names[:cut_off],
    "product_description" : product_descriptions[:cut_off],
    "colour" : colours[:cut_off],
    "type" : types[:cut_off],
    "vintage" : vintages[:cut_off],
    "state" : states[:cut_off],
    "region" : regions[:cut_off],
    "winery" : wineries[:cut_off],
    "size" : sizes[:cut_off],
    "non_member_price" : non_member_prices[:cut_off],
    "member_price" : member_prices[:cut_off],
    "body" : bodies[:cut_off],
    "sweetness" : sweetnesses[:cut_off]
}

populated_wine_df = pd.DataFrame(populated_wine_df)

In [7]:
populated_wine_df

,href,product_brand,product_name,product_description,colour,type,vintage,state,region,winery,size,non_member_price,member_price,body,sweetness
0,https://www.danmurphys.com.au/product/DM_10851...,Mumm,Grand Cordon Rosé Champagne,"Appetizingly elegant and intense, the wine del...",Champagne-Sparkling,Pinot Noir,Non Vintage,NaN,Champagne,NaN,750mL,99.99,NaN,NaN,Dry
1,https://www.danmurphys.com.au/product/DM_10750...,NaN,NaN,NaN,Champagne-Sparkling,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
populated_wine_df.iloc[0]

href                   https://www.danmurphys.com.au/product/DM_10851...
product_brand                                                       Mumm
product_name                                 Grand Cordon Rosé Champagne
product_description    Appetizingly elegant and intense, the wine del...
colour                                               Champagne-Sparkling
type                                                          Pinot Noir
vintage                                                      Non Vintage
state                                                                NaN
region                                                         Champagne
winery                                                               NaN
size                                                               750mL
non_member_price                                                   99.99
member_price                                                         NaN
body                                               

In [8]:
# save df to csv
file_path = f"data/all_wine_data_subsets/wine_dataset_i_{starting_i}-{i-1}.csv"
populated_wine_df.to_csv(file_path, index=False)